In [1]:
import os
import gc
import re
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.layers import (
    Input, Dense, Reshape, Flatten, Concatenate,
    Dropout, BatchNormalization, LayerNormalization,
    Add, Layer
    )
from keras.preprocessing import image
from transformers import AutoTokenizer, TFAutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')
tqdm.pandas()

2025-10-12 18:28:20.859924: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760293701.078462      75 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760293701.143499      75 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# For reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seed(42)

In [4]:
class Config:
    N_SPLITS = 5
    
    OUTPUT_DIR = os.getcwd()
    PREPROCESSED_IMAGE_DIR = os.path.join(OUTPUT_DIR, 'preprocessed_images')
    TRAIN_DIR = "/kaggle/input/amazon-ml-challenge-25/train.csv"
    TEST_DIR = "/kaggle/input/amazon-ml-challenge-25/test.csv"
    IMAGES_TRAIN_DIR = "/kaggle/input/amazon-ml-images-50-sample/optimized_50_percent_sample/train"
    IMAGES_TEST_DIR = "/kaggle/input/amazon-ml-images-50-sample/optimized_50_percent_sample/test"
    LOCAL_MODEL_PATH = ""
    
    CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
    IMG_SIZE = 224
    MAX_TEXT_LEN = 77  # CLIP's max sequence length
    
    BATCH_SIZE = 32
    EPOCHS = 8
    LEARNING_RATE = 1e-4
    
    MAX_FEATURES_TAGS = 500  # Max features for TF-IDF on tags
    MAX_FEATURES_CATS = 200  # Max features for TF-IDF on category candidates

CONFIG = Config()

In [5]:
print("1. Loading data...")
train_df = pd.read_csv(CONFIG.TRAIN_DIR)
test_df = pd.read_csv(CONFIG.TEST_DIR)

1. Loading data...


In [6]:
train_image_paths = train_df['sample_id'].apply(lambda x : f"{os.path.join(CONFIG.IMAGES_TRAIN_DIR, str(x))}.jpg").tolist()
test_image_paths = test_df['sample_id'].apply(lambda x : f"{os.path.join(CONFIG.IMAGES_TEST_DIR, str(x))}.jpg").tolist()

In [7]:
# base_name = lambda file : ".".join(file.split(".")[:-1])
# train_samples = [int(base_name(name)) for name in os.listdir(CONFIG.IMAGES_TRAIN_DIR)]
# test_samples = [int(base_name(name)) for name in os.listdir(CONFIG.IMAGES_TEST_DIR)]

In [8]:
# def sample_data(df, samples) :
#     data = []
#     for sample in samples :
#         data.append(df[df["sample_id"] == sample].values[0])
#     return pd.DataFrame(data, columns = df.columns)

# train_df = sample_data(train_df, train_samples)
# test_df = sample_data(test_df, test_samples)

In [9]:
def extract_features(text: str) -> dict:
    if not isinstance(text, str):
        text = ""  # Handle potential NaN values

    features = {}
    lower_text = text.lower()

    # Pack Count (IPQ)
    pack_match = re.search(r'(?:pack of|set of|pk of|pack|pk)\s*(\d+)|(\d+)\s*(?:pack|pk|count|ct)', text, flags=re.I)
    features['pack_count'] = int(pack_match.group(1) or pack_match.group(2)) if pack_match else 1

    # Numeric Quantity and Unit (e.g., "16.5 oz", "2 lbs")
    qty_match = re.search(r'(\d+(?:\.\d+)?)\s*(oz|ounce|lb|pound|g|kg|ml|l|fl oz|fl\. oz\.)\b', text, flags=re.I)
    if qty_match:
        features['numeric_quantity'] = float(qty_match.group(1))
        features['quantity_unit'] = qty_match.group(2).lower().replace('.', '')
    else:
        features['numeric_quantity'] = np.nan
        features['quantity_unit'] = 'unknown'

    # Brand (more robustly captures brands with numbers or multiple words)
    brand_match = re.search(r"Item Name:\s*([A-Z0-9][A-Za-z0-9' -]{1,30})\b", text)
    features['brand'] = brand_match.group(1).strip() if brand_match else 'Unknown'

    attribute_keywords = [
        'premium', 'organic', 'gourmet', 'heavy-duty', 'professional', 'industrial',
        'handmade', 'natural', 'usda', 'gluten-free', 'non-gmo', 'eco-friendly',
        'wireless', 'bluetooth', 'smart', 'hd', '4k', 'waterproof'
    ]
    for keyword in attribute_keywords:
        features[f'attr_{keyword.replace("-", "_")}'] = 1 if keyword in lower_text else 0

    materials = ['wood', 'steel', 'stainless steel', 'leather', 'cotton', 'plastic', 'ceramic', 'glass', 'aluminum', 'copper']
    for material in materials:
        features[f'mat_{material.replace(" ", "_")}'] = 1 if material in lower_text else 0

    colors = ['red', 'blue', 'black', 'white', 'green', 'yellow', 'silver', 'gold', 'brown', 'purple', 'orange']
    for color in colors:
        features[f'color_{color}'] = 1 if color in lower_text else 0

    features['text_length'] = len(text)
    features['word_count'] = len(text.split())
    features['bullet_point_count'] = text.count("Bullet Point")
    
    desc_match = re.search(r"Product Description:\s*(.+)", text, flags=re.S | re.I)
    features['description_length'] = len(desc_match.group(1).strip()) if desc_match else 0
    
    return features

In [10]:
print("2. Applying feature engineering...")
train_features_df = train_df['catalog_content'].progress_apply(extract_features).apply(pd.Series)
test_features_df = test_df['catalog_content'].progress_apply(extract_features).apply(pd.Series)

2. Applying feature engineering...


  0%|          | 0/75000 [00:00<?, ?it/s]

  0%|          | 0/75000 [00:00<?, ?it/s]

In [11]:
from sklearn.preprocessing import FunctionTransformer

print("3. Vectorizing engineered features...")

numerical_cols = [
    'pack_count', 'numeric_quantity', 'text_length', 
    'word_count', 'bullet_point_count', 'description_length'
] + [col for col in train_features_df.columns if col.startswith(('attr_', 'mat_', 'color_'))]

# High-cardinality categorical columns (many unique values)
high_card_categorical_cols = ['brand']

# Low-cardinality categorical columns (few unique values)
low_card_categorical_cols = ['quantity_unit']

# Pipeline for numerical features:
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipelinea for high-cardinality text features (e.g., Brand):
brand_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('flatten', FunctionTransformer(lambda x : x.ravel(), validate = False)),
    ('tfidf', TfidfVectorizer(max_features=500, token_pattern=r'\b[a-zA-Z0-9-]+\b'))
])

# --- Use ColumnTransformer to apply pipelines to the correct columns ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_cols),
        ('brand_tfidf', brand_pipeline, ['brand']),
        ('unit_onehot', OneHotEncoder(handle_unknown='ignore'), low_card_categorical_cols)
    ],
    remainder='drop'
)

print("   - Fitting preprocessor and transforming data...")
engineered_train_feats = preprocessor.fit_transform(train_features_df)
engineered_test_feats = preprocessor.transform(test_features_df)

if hasattr(engineered_train_feats, "toarray"):
    engineered_train_feats = engineered_train_feats.toarray()
    engineered_test_feats = engineered_test_feats.toarray()

print(f"\nSuccessfully created engineered feature matrices.")
print(f"Engineered training feature shape: {engineered_train_feats.shape}")
print(f"Engineered test feature shape: {engineered_test_feats.shape}")


3. Vectorizing engineered features...
   - Fitting preprocessor and transforming data...

Successfully created engineered feature matrices.
Engineered training feature shape: (75000, 555)
Engineered test feature shape: (75000, 555)


In [12]:
from huggingface_hub import snapshot_download

local_dir = "clip-model-local"

os.makedirs(local_dir, exist_ok=True)

try:
    snapshot_download(
        repo_id=CONFIG.CLIP_MODEL_NAME,
        local_dir=local_dir,
        local_dir_use_symlinks=False, # Set to False to avoid issues on some systems
        resume_download=True
    )
    CONFIG.LOCAL_MODEL_PATH = os.path.abspath(local_dir)
    print("Download complete!")
    print(f"Model files are saved in: {CONFIG.LOCAL_MODEL_PATH}")

except Exception as e:
    print(f"An error occurred during download: {e}")
    print("Please check your internet connection and firewall settings.")

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

flax_model.msgpack:   0%|          | 0.00/605M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Download complete!
Model files are saved in: /kaggle/working/clip-model-local


In [13]:
print("4. Preparing text and image data...")
# --- Text Tokenization for CLIP ---
tokenizer = AutoTokenizer.from_pretrained(CONFIG.LOCAL_MODEL_PATH)
train_text_tokens = tokenizer(text=train_df['catalog_content'].fillna("").tolist(), return_tensors='np', max_length=CONFIG.MAX_TEXT_LEN, padding='max_length', truncation=True)
test_text_tokens = tokenizer(text=test_df['catalog_content'].fillna("").tolist(), return_tensors='np', max_length=CONFIG.MAX_TEXT_LEN, padding='max_length', truncation=True)

4. Preparing text and image data...


In [14]:
# --- Image Preprocessing & Caching ---
os.makedirs(CONFIG.PREPROCESSED_IMAGE_DIR, exist_ok=True)
def load_and_preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    try:
        img = tf.io.decode_jpeg(img, channels=3)
    except tf.errors.InvalidArgumentError:
        # If decode fails, return a black image
        print(image_path, end = ", ")
        return tf.zeros([CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3])
        
    img = tf.image.resize(img, [CONFIG.IMG_SIZE, CONFIG.IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3])
    return img

In [15]:
def create_tf_dataset(image_paths, text_tokens, engineered_feats, labels=None, shuffle=False):
    print("[WARN] These images did not load")
    print("[", end = "")
    image_ds = tf.data.Dataset.from_tensor_slices(image_paths).map(
        load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE
    )
    print("]")
    
    text_ds = tf.data.Dataset.from_tensor_slices(
        (text_tokens['input_ids'], text_tokens['attention_mask'])
    )
    engineered_ds = tf.data.Dataset.from_tensor_slices(engineered_feats)

    if labels is not None:
        labels_ds = tf.data.Dataset.from_tensor_slices(labels)
        full_ds = tf.data.Dataset.zip(((image_ds, text_ds, engineered_ds), labels_ds))
    else:
        full_ds = tf.data.Dataset.zip((image_ds, text_ds, engineered_ds))

    if shuffle:
        full_ds = full_ds.shuffle(buffer_size=1024)

    # Batch and prefetch for performance
    full_ds = full_ds.batch(CONFIG.BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

    # Reformat to the dictionary structure the model expects
    if labels is not None:
        full_ds = full_ds.map(lambda x, y: ({
            'image_input': x[0],
            'text_ids_input': x[1][0],
            'text_mask_input': x[1][1],
            'engineered_input': x[2]
        }, y))
    else:
        full_ds = full_ds.map(lambda x: {
            'image_input': x[0],
            'text_ids_input': x[1][0],
            'text_mask_input': x[1][1],
            'engineered_input': x[2]
        })
        
    return full_ds

In [16]:
class MultiHeadCrossAttention(Layer):
    def __init__(self, d_model=512, num_heads=8, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_dense = Dense(d_model)
        self.key_dense = Dense(d_model)
        self.value_dense = Dense(d_model)
        self.combine_heads = Dense(d_model)
        self.layernorm = LayerNormalization()
        self.add = Add()

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.head_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, query_input, key_input, value_input):
        batch_size = tf.shape(query_input)[0]
        query = self.query_dense(query_input)
        key = self.key_dense(key_input)
        value = self.value_dense(value_input)
        
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)
        
        attention_output = self.attention(query, key, value)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (batch_size, -1, self.d_model))
        
        combined = self.combine_heads(concat_attention)
        # Add & Norm
        output = self.layernorm(self.add([query_input, combined]))
        return output

In [ ]:
def create_model():
    # --- Define Inputs ---
    image_input = Input(shape=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), name="image_input")
    text_ids_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_ids_input")
    text_mask_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_mask_input")
    engineered_input = Input(shape=(engineered_train_feats.shape[1],), name="engineered_input")
    
    # --- Load Pre-trained CLIP Model ---
    clip_model = TFAutoModel.from_pretrained(CONFIG.CLIP_MODEL_NAME)
    
    # Freeze CLIP
    clip_model.clip.text_model.trainable = False
    clip_model.clip.vision_model.trainable = False
    
    # --- Get Embeddings ---
    image_embeds = clip_model.get_image_features(pixel_values=image_input, training=False)
    text_embeds = clip_model.get_text_features(input_ids=text_ids_input, attention_mask=text_mask_input, training=False)

    # Add a sequence dimension for attention
    text_embeds_seq = Reshape((1, -1))(text_embeds)
    image_embeds_seq = Reshape((1, -1))(image_embeds)

    # --- Multi-Head Cross-Modal Attention Block ---
    attention_layer = MultiHeadCrossAttention(d_model=text_embeds.shape[-1], num_heads=8)
    text_to_image_features = attention_layer(text_embeds_seq, image_embeds_seq, image_embeds_seq)
    image_to_text_features = attention_layer(image_embeds_seq, text_embeds_seq, text_embeds_seq)
    
    fused_features = Concatenate()([
        Flatten()(text_to_image_features),
        Flatten()(image_to_text_features)
    ])
    
    # --- Final Fusion and Regression Head ---
    all_features = Concatenate()([fused_features, engineered_input])
    
    x = BatchNormalization()(all_features)
    x = Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    x = Dropout(0.5)(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    
    output = Dense(1, activation='relu', name='price_output')(x) # ReLU to ensure positive price
    
    model = keras.Model(
        inputs=[image_input, text_ids_input, text_mask_input, engineered_input],
        outputs=output
    )
    
    return model

In [85]:
# Custom SMAPE Metric for monitoring
def smape_metric(y_true, y_pred):
    y_true = tf.expm1(y_true) # Reverse log transform
    y_pred = tf.expm1(y_pred)
    numerator = tf.abs(y_pred - y_true)
    denominator = (tf.abs(y_true) + tf.abs(y_pred)) / 2.0
    return tf.reduce_mean(numerator / (denominator + 1e-8)) * 100.0

In [78]:
print("5. Preparing data paths and arrays for the tf.data pipeline...")

all_train_text_tokens = {
    'input_ids': train_text_tokens['input_ids'],
    'attention_mask': train_text_tokens['attention_mask']
}
y_log_full = np.log1p(train_df["price"])

# The actual test data for final prediction
test_data_for_pipeline = {
    'image_paths': test_image_paths,
    'text_tokens': test_text_tokens,
    'engineered_feats': engineered_test_feats
}

train_df['price_bin'] = pd.cut(train_df['price'], bins=10, labels=False, include_lowest=True)
train_df["price_bin"] = np.where(train_df["price_bin"].value_counts()[train_df["price_bin"]] <= 2, 2, train_df["price_bin"])

indices = np.arange(len(train_df))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2, # Using 20% for validation
    random_state=42,
    stratify=train_df['price_bin']
)

print("\nCreating training and validation datasets...")

# Training dataset
train_ds = create_tf_dataset(
    image_paths=np.array(train_image_paths)[train_idx],
    text_tokens={'input_ids': all_train_text_tokens['input_ids'][train_idx], 
                 'attention_mask': all_train_text_tokens['attention_mask'][train_idx]},
    engineered_feats=engineered_train_feats[train_idx],
    labels=y_log_full[train_idx],
    shuffle=True
)

# Validation dataset
val_ds = create_tf_dataset(
    image_paths=np.array(train_image_paths)[val_idx],
    text_tokens={'input_ids': all_train_text_tokens['input_ids'][val_idx], 
                 'attention_mask': all_train_text_tokens['attention_mask'][val_idx]},
    engineered_feats=engineered_train_feats[val_idx],
    labels=y_log_full[val_idx]
)

print(f"Datasets created. Training on {len(train_idx)} samples, validating on {len(val_idx)} samples.")

5. Preparing data paths and arrays for the tf.data pipeline...

Creating training and validation datasets...
[WARN] These images did not load
[

I0000 00:00:1760284864.697845      79 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1760284864.698584      79 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


]
[WARN] These images did not load
[]
Datasets created. Training on 60000 samples, validating on 15000 samples.


In [88]:
keras.backend.clear_session()
model = create_model()
optimizer = keras.optimizers.AdamW(learning_rate=CONFIG.LEARNING_RATE)
model.compile(optimizer=optimizer, loss='huber', metrics=[smape_metric])

# --- Callbacks ---
lr_reducer = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)

print("\n===== VALIDATION TRAINING =====")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG.EPOCHS,
    callbacks=[lr_reducer, early_stopper]
)

All model checkpoint layers were used when initializing TFCLIPModel.

All the layers of TFCLIPModel were initialized from the model checkpoint at openai/clip-vit-base-patch32.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFCLIPModel for predictions without further training.


ValueError: Data of type <class 'keras.src.backend.common.keras_tensor.KerasTensor'> is not allowed only (<class 'tensorflow.python.framework.tensor.Tensor'>, <class 'bool'>, <class 'int'>, <class 'transformers.utils.generic.ModelOutput'>, <class 'tuple'>, <class 'list'>, <class 'dict'>, <class 'numpy.ndarray'>) is accepted for pixel_values.

In [100]:
from tensorflow.keras.layers import Lambda
# --- Define Inputs ---
image_input = Input(shape=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), name="image_input")
text_ids_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_ids_input")
text_mask_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_mask_input")
engineered_input = Input(shape=(engineered_train_feats.shape[1],), name="engineered_input")

# --- Load Pre-trained CLIP Model ---
clip_model = TFAutoModel.from_pretrained(CONFIG.CLIP_MODEL_NAME)

def clip_forward(inputs):
    pixel_values, input_ids, attention_mask = inputs
    outputs = clip_model({
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask
    })
    return outputs.image_embeds, outputs.text_embeds

image_embeds, text_embeds = Lambda(clip_forward, output_shape=[(512,), (512,)])([image_input, text_ids_input, text_mask_input])

# class ClipEmbeddingLayer(Layer):
#     """
#     Custom Keras layer to wrap the Hugging Face CLIP model.
    
#     This layer handles:
#     1. Loading the pre-trained CLIP model once during initialization.
#     2. Correctly passing multiple inputs (image, text) to the CLIP model.
#     3. Explicitly defining the output shape for perfect integration with the Keras graph.
#     """
#     def __init__(self, clip_model_path, **kwargs):
#         super().__init__(**kwargs)
#         # Load the model during layer initialization
#         self.clip_model = TFAutoModel.from_pretrained(clip_model_path)
#         # Freeze the model
#         self.clip_model.trainable = False
        
#     def call(self, inputs):
#         """
#         This is the forward pass of the layer.
#         """
#         # The inputs will be a list of tensors from the Keras functional API
#         image_input, text_ids_input, text_mask_input = inputs
        
#         # Pass the inputs correctly to the clip_model using a dictionary
#         clip_outputs = self.clip_model({
#             "pixel_values": image_input,
#             "input_ids": text_ids_input,
#             "attention_mask": text_mask_input
#         })
        
#         # Return the two embeddings as a tuple
#         return clip_outputs.image_embeds, clip_outputs.text_embeds

#     def compute_output_shape(self, input_shape):
#         """
#         This method is crucial. It explicitly tells Keras the shape of the output,
#         solving the shape inference problem that breaks BatchNormalization.
#         """
#         # input_shape is a list of shapes for [image, ids, mask]
#         batch_size = input_shape[0][0] # Get batch size from the image input shape
        
#         # CLIP base model outputs 512-dimensional embeddings
#         embedding_dim = 512 
        
#         # We return a tuple of two shapes, one for each output tensor
#         return ((batch_size, embedding_dim), (batch_size, embedding_dim))


# image_input = Input(shape=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), name="image_input")
# text_ids_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_ids_input")
# text_mask_input = Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_mask_input")
# engineered_input = Input(shape=(engineered_train_feats.shape[1],), name="engineered_input")

# # --- Instantiate our custom CLIP layer ---
# embedding_layer = ClipEmbeddingLayer("./clip-model-local")

# # --- Get Embeddings by calling our custom layer ---
# # It correctly handles the call and returns two separate, correctly-shaped tensors
# image_embeds, text_embeds = embedding_layer([image_input, text_ids_input, text_mask_input])

# --- The rest of your architecture can now proceed safely ---
# Add a sequence dimension for attention
text_embeds_seq = Reshape((1, -1))(text_embeds)
image_embeds_seq = Reshape((1, -1))(image_embeds)

# --- Multi-Head Cross-Modal Attention Block ---
attention_layer = MultiHeadCrossAttention(d_model=512, num_heads=8) # Using 512 directly
text_to_image_features = attention_layer(text_embeds_seq, image_embeds_seq, image_embeds_seq)
image_to_text_features = attention_layer(image_embeds_seq, text_embeds_seq, text_embeds_seq)

fused_features = Concatenate()([
    Flatten()(text_to_image_features),
    Flatten()(image_to_text_features)
])

# --- Final Fusion and Regression Head ---
all_features = Concatenate()([fused_features, engineered_input])

# BatchNormalization will now work because Keras knows the exact input shape
x = BatchNormalization()(all_features)
x = Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)
x = Dropout(0.5)(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)

output = Dense(1, activation='relu', name='price_output')(x)

model = Model(
    inputs=[image_input, text_ids_input, text_mask_input, engineered_input],
    outputs=output
)

All model checkpoint layers were used when initializing TFCLIPModel.

All the layers of TFCLIPModel were initialized from the model checkpoint at openai/clip-vit-base-patch32.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFCLIPModel for predictions without further training.


ValueError: Shapes used to initialize variables must be fully-defined (no `None` dimensions). Received: shape=(None,) for variable path='batch_normalization_2/gamma'

In [ ]:
# Evaluate the model on the validation set
val_loss, val_smape = model.evaluate(val_ds)
print(f"Validation SMAPE from the best epoch: {val_smape:.4f}")

# Determine the optimal number of epochs for retraining
optimal_epochs = len(history.history['val_loss']) - early_stopper.patience + 1
print(f"Optimal number of epochs found: {optimal_epochs}")

In [ ]:
print("5. Starting training with K-Fold Cross-Validation...")
kf = KFold(n_splits=CONFIG.N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(tqdm(kf.split(train_df), desc = "SKF")):
    print(f"\n===== FOLD {fold+1}/{CONFIG.N_SPLITS} =====")
    
    # --- Prepare Data for Fold ---
    X_train = {
        'image_input': train_images[train_idx],
        'text_ids_input': train_text_tokens['input_ids'][train_idx],
        'text_mask_input': train_text_tokens['attention_mask'][train_idx],
        'engineered_input': engineered_train_feats[train_idx]
    }
    y_train_fold = y_log[train_idx]
    
    X_val = {
        'image_input': train_images[val_idx],
        'text_ids_input': train_text_tokens['input_ids'][val_idx],
        'text_mask_input': train_text_tokens['attention_mask'][val_idx],
        'engineered_input': engineered_train_feats[val_idx]
    }
    y_val_fold = y_log[val_idx]
    
    # --- Build and Compile Model ---
    keras.backend.clear_session()
    model = create_model()
    optimizer = keras.optimizers.AdamW(learning_rate=CONFIG.LEARNING_RATE)
    # Huber loss is a good proxy for MAE, robust to outliers
    model.compile(optimizer=optimizer, loss='huber', metrics=[smape_metric])
    
    # --- Callbacks ---
    lr_reducer = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
    
    # --- Train Model ---
    model.fit(
        X_train, y_train_fold,
        validation_data=(X_val, y_val_fold),
        epochs=CONFIG.EPOCHS,
        batch_size=CONFIG.BATCH_SIZE,
        callbacks=[lr_reducer, early_stopper]
    )
    
    # --- Predict and Store ---
    oof_preds[val_idx] = model.predict(X_val).flatten()
    
    X_test = {
        'image_input': test_images,
        'text_ids_input': test_text_tokens['input_ids'],
        'text_mask_input': test_text_tokens['attention_mask'],
        'engineered_input': engineered_test_feats
    }
    test_preds += model.predict(X_test).flatten() / CONFIG.N_SPLITS
    
    # --- Clean up ---
    del model
    gc.collect()
print("\nTraining complete.")

In [ ]:
# Reverse log transform for final predictions
oof_preds_final = np.expm1(oof_preds)
test_preds_final = np.expm1(test_preds)

# Ensure no negative prices
test_preds_final[test_preds_final < 0] = 0 

# Calculate overall OOF SMAPE
final_oof_smape = smape_metric(y, oof_preds_final).numpy()
print(f"Overall Out-of-Fold SMAPE: {final_oof_smape:.4f}")

# --- Create Submission File ---
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_preds_final
})
submission_df.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' created successfully.")
print("Top 5 predictions:")
print(submission_df.head())